# 4: Investigating Output

**BEFORE BEGINNING THIS EXERCISE** -  Check that your kernel (upper right corner, above) is `NPL 2023a`. This should be the default kernel, but if it is not, click on that button and select `NPL 2023a`.

_______________
This activity was developed primarily by Adrianna Foster.

## CLM output

While CLM-FATES and regular CLM share many output variables (e.g. `ASA`, `FSR`, `FSH`, etc.), most of the vegetation-related variables are now being simulated, calculated, and output by FATES. The variable names and their structure (as well as the PFTs!) are slightly different from a "big leaf" CLM run.

Here we will show you how to look at some basic FATES output for the global cases you just ran.

*As before, start by loading some packages*

In [ ]:
import os
import glob
import xarray as xr
import functools

%matplotlib inline

### Helper functions

These are a few helper functions that we will use in this notebook.

In [ ]:
def preprocess(data_set: xr.Dataset, data_vars: list[str]) -> xr.Dataset:
    """Preprocesses an xarray Dataset by subsetting to specific variables - to be used on read-in

    Args:
        data_set (xr.Dataset): input Dataset

    Returns:
        xr.Dataset: output Dataset
    """

    return data_set[data_vars]


def annual_sum(raw_values: xr.DataArray, conversion_factor: float = 1.0) -> xr.DataArray:
    """Computes annual sum

    Args:
        raw_values (xr.DataArray): input raw data
        conversion_factor (float, optional): conversion factor. Defaults to 1.0

    Returns:
        xr.DataArray: annual sum output
    """

    months = raw_values["time.daysinmonth"]
    return conversion_factor * (months * raw_values).groupby("time.year").sum()


def annual_mean(raw_values: xr.DataArray, conversion_factor: float = 1.0) -> xr.DataArray:
    """Computes weighted annual mean using daysinmonth for missing-aware inputs.

    Args:
        raw_values (xr.DataArray): input raw data
        conversion_factor (float, optional): conversion factor. Defaults to 1.0

    Returns:
        xr.DataArray: annual mean output
    """

    months = raw_values["time.daysinmonth"]
    
    # multiply by number of days in month and conversion factor
    weighted = (raw_values * conversion_factor) * months

    # compute number of valid days per year
    valid_days = months.where(raw_values.notnull())

    # group and sum weighted data and valid days
    ann_sum = weighted.groupby("time.year").sum(dim="time", skipna=True)
    days_per_year = valid_days.groupby("time.year").sum(dim="time", skipna=True)

    return ann_sum / days_per_year.where(days_per_year > 0.0)

## 1. Reading and formatting data

**Note**: the drop-down solutions, below, assume you used i.fates.year1.a and i.fates.year1.a_vcmax output for plotting for this section.

### 1.1 Read in the data
The first step is to grab the history files from the runs you completed in the FATES challenge.

For this example we will use:
- gross primary production (`FATES_GPP` and `FATES_GPP_PF`), 
- latent heat flux (`EFLX_LH_TOT`), and
- sensible heat flux (`FSH`)

<b>NOTE:</b> These are the raw history files that CTSM writes out. 

By default, they include grid cell averaged monthly means for different state and flux variables.

<div class="alert alert-block alert-info">
    <b>TIP:</b> If you want to look at other variables, the <b>data_vars</b> variable in the cell below is where you can modify what we're reading off of the CLM history files.
</div>

#### Printing information about the dataset is helpful for understanding your data. 
- *What dimensions do your data have?*
- *What are the coordinate variables?*
- *What variables are we looking at?*
- *Is there other helpful information, or are there attributes in the dataset we should be aware of?*


In [ ]:
# note: change the user here to your user name
user = 'afoster'

# this should be the history directory for the control directory and the one with vcmax updated
control_hist = f"/glade/derecho/scratch/{user}/archive/i.fates.year1.a/lnd/hist"
vcmax_hist = f"/glade/derecho/scratch/{user}/archive/i.fates.year1.a_vcmax/lnd/hist"

# create path to all files, including unix wild card for all dates
control_files = sorted(glob.glob(os.path.join(control_hist, "i.fates.year1.a.clm2.h0a.*")))
vcmax_files = sorted(glob.glob(os.path.join(vcmax_hist, "i.fates.year1.a_vcmax.clm2.h0a.*")))

# read in files as xarray datasets:
data_vars = ['FATES_GPP', 'FATES_GPP_PF', 'EFLX_LH_TOT', 'FSH',
                  'FATES_FRACTION', 'area', 'landfrac']

ds_control = xr.open_mfdataset(
    control_files,
    preprocess=functools.partial(preprocess, data_vars=data_vars)
)
ds_vcmax = xr.open_mfdataset(
    vcmax_files, 
    preprocess=functools.partial(preprocess, data_vars=data_vars)
)

In [ ]:
# print information about one of the datasets
ds_control

You can also print information about the variables in your dataset. The example below prints information about one of the data variables (`FATES_GPP_PF`) we read in. You can modify this cell to look at some of the other variables in the dataset.

*What are the units, long name, and dimensions of your data?*

In [ ]:
ds_control.FATES_GPP_PF

### 1.2 Simple Calculations
To begin with, we need to modify all `FATES_` variables by multiplying them by `FATES_FRACTION`. This is because the FATES model doesn't know how much of each gridcell is occupied by natural vegetation, and so the grid-cell averaging assumings all of it is. We need to correct for this by multiplying by the variable `FATES_FRACTION`:

In [ ]:
ds_control['GPP'] = ds_control.FATES_GPP*ds_control.FATES_FRACTION
ds_control['GPP'].attrs = ds_control.FATES_GPP.attrs

ds_control['GPP_PF'] = ds_control.FATES_GPP_PF*ds_control.FATES_FRACTION
ds_control['GPP_PF'].attrs = ds_control.FATES_GPP_PF.attrs

ds_vcmax['GPP'] = ds_vcmax.FATES_GPP*ds_vcmax.FATES_FRACTION
ds_vcmax['GPP'].attrs = ds_vcmax.FATES_GPP.attrs

ds_vcmax['GPP_PF'] = ds_vcmax.FATES_GPP_PF*ds_vcmax.FATES_FRACTION
ds_vcmax['GPP_PF'].attrs = ds_vcmax.FATES_GPP_PF.attrs

---
## 2. Basic plotting with global FATES
### 2.1 Easy plots using xarray
To get a first look at the data, we can plot a year of data from the simulation.

<div class="alert alert-block alert-info">
    <b>NOTE:</b> The plotting function only works with 1D or 2D data. Our data are 3D (time, lat, lon), so we need to specify a specific value `x`, `y`, and `col`.
</div>

- We will plot GPP (variable = `GPP`). Note that we select the variable by specifying our dataset, `ds_control`, and the variable. 
- This plotting function will plot `GPP` for each simulation in our dataset, and we have set the column to be month (`col="time"`).

*More plotting examples are on the [xarray web site](https://docs.xarray.dev/en/latest/user-guide/plotting.html)*

In [ ]:
ds_control.GPP.plot(x='lon', y='lat', col="time", col_wrap=6);

<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman">Click here for the solution</font></summary><br>

![plot example](../../../images/diagnostics/clm_ctsm/fates_plot_monthly_gpp.png)

*<p style="text-align: left;"> Figure: Plotting solution for monthly GPP for our control simulation. </p>*
    
</details>
</div>

#### Question

What do you notice in the Northern Hemisphere over the course of the year?

### 2.2 Calculating differences

We can calculate the differences between our control run with FATES and the one in which we updated the vcmax parameter for PFT 1.

The below code:
- Calculates annual GPP for both simulations
- Defines the difference as a new variable, `gpp_diff`

In [ ]:
gpp_cf = 24*60*60 # kgC/m2/yr

In [ ]:
gpp_ann_control = annual_sum(ds_control.GPP, gpp_cf)*ds_control.landfrac.isel(time=0)
gpp_ann_vcmax = annual_sum(ds_vcmax.GPP, gpp_cf)*ds_vcmax.landfrac.isel(time=0)
gpp_diff = gpp_ann_vcmax - gpp_ann_control
gpp_diff.attrs = {'units': 'kgC m-2 yr-1', 'long_name': 'annual gross primary production difference'}
gpp_diff.plot(cmap='RdBu_r', vmin=-1.5, vmax=1.5);

<div class="alert alert-success">
<details>
<summary><font face="Times New Roman">Click here for the solution</font></summary><br>

![plot example](../../../images/diagnostics/clm_ctsm/fates_annual_gpp_diff.png)

*<p style="text-align: left;"> Figure: Plotting solution for annual GPP for our updated solution minus the control simulation. </p>*
    
</details>
</div>

#### Questions
- How is GPP different for our updated simulation, relative to the control simulation?
- Where are the differences the largest?
- Where are they absent?
- What is causing these GPP changes in different regions and why is it only in some areas?

#### Do the differences affect all PFTs?
To find out, we can plot the `GPP_PF` variable, which is indexed by PFT.

We will also grab June (`.isel(time=5)`; 0-indexed):

In [ ]:
ds_control.GPP_PF.isel(time=5).plot(x='lon', y='lat', col="fates_levpft", col_wrap=6);

<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman">Click here for the solution</font></summary><br>

![plot example](../../../images/diagnostics/clm_ctsm/fates_monthly_gpp_pft.png)

*<p style="text-align: left;"> Figure: Plotting solution for comparing GPP by PFT for our updated simulation from our control. </p>*
    
</details>
</div>

#### Do the differences
To find out, do the same calculation as above, but with the `GPP_PF` variable.

In [ ]:
gpp_ann_control_pf = annual_sum(ds_control.GPP_PF, gpp_cf)*ds_control.landfrac.isel(time=0)
gpp_ann_vcmax_pf = annual_sum(ds_vcmax.GPP_PF, gpp_cf)*ds_vcmax.landfrac.isel(time=0)
gpp_diff_pf = gpp_ann_vcmax_pf - gpp_ann_control_pf
gpp_diff_pf.attrs = {'units': 'kgC m-2 yr-1', 'long_name': 'annual gross primary production difference'}
gpp_diff_pf.plot(x='lon', y='lat', col="fates_levpft", col_wrap=6)

<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman">Click here for the solution</font></summary><br>

![plot example](../../../images/diagnostics/clm_ctsm/fates_annual_gpp_diff_pft.png)

*<p style="text-align: left;"> Figure: Plotting solution for comparing GPP by PFT for our updated simulation from our control. </p>*
    
</details>
</div>

#### Questions:
- Which PFT(s) show any difference? Why is this?
- Why don't other PFTs show any differences? Shouldn't there be some kind of compensating factor?

*<b>Hint</b>: you can check the `fates_pftname` variable in `/glade/u/home/$USER/code/my_cesm_code/CTSM/src/fates/parameter_files/fates_params_default.json` to see which PFT corresponds to which index.*

#### Do the differences affect other variables?
To find out, we can plot `EFLX_LH_TOT`, which is latent heat flux. How do you think this will be affected?

`EFLX_LH_TOT` is a regular CLM variable (that is, it is calculated by CLM, not FATES, so we should not multiply it by `FATES_FRACTION`).

In [ ]:
ds_control.EFLX_LH_TOT

#### Do the differences
Let's calculate annual latent heat flux. This time we will use annual mean instead of an annual sum:

In [ ]:
lh_ann_control = annual_mean(ds_control.EFLX_LH_TOT)*ds_control.landfrac.isel(time=0)
lh_ann_vcmax = annual_mean(ds_vcmax.EFLX_LH_TOT)*ds_vcmax.landfrac.isel(time=0)
lh_diff = lh_ann_vcmax - lh_ann_control
lh_diff.attrs = {'units': 'W m-2', 'long_name': 'annual latent heat flux difference'}
lh_diff.plot(vmin=-20, vmax=20, cmap='RdBu_r')

<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman">Click here for the solution</font></summary><br>

![plot example](../../../images/diagnostics/clm_ctsm/fates_annual_lh_diff.png)

*<p style="text-align: left;"> Figure: Plotting solution for comparing latent heat flux for our updated simulation from our control. </p>*
    
</details>
</div>

#### Questions:
- How is latent heat flux different?
- Try looking at sensible heat flux (`FSH`). What differences do you see? 

*Note that you might want to change the minimun (`vmin`) and maximum (`vmax`) colorbar values for the plot when you change variables*